# Caso 11 - Predicción de Precios de Arriendo Mediante Modelos de Regresión. ¿Cómo capturar relaciones no lineales en mis datos?

En este caso de estudio, trabajaremos con una base de datos que contiene información sobre propiedades en arriendo, con el objetivo de predecir el precio de arriendo a partir de variables relacionadas con las características del inmueble. Las variables disponibles incluyen la ubicación, número de habitaciones, baños, área construida, y otras características relevantes de la propiedad.

Para abordar el problema, aplicaremos técnicas de regresión supervisada utilizando modelos como **Regresión Lineal**, **Gradient Boosting** y **K-Nearest Neighbors (KNN)**. Previamente, se realizará un proceso de preprocesamiento y transformación de variables usando herramientas de la librería `scikit-learn`, como codificación de variables categóricas, escalamiento de variables numéricas y manejo de valores faltantes.

También se evaluará el impacto de las variables en los modelos, especialmente en la regresión lineal y el modelo de Gradient Boosting, que permiten interpretar la importancia relativa de cada variable predictora. Finalmente, se comparará el desempeño de los modelos mediante métricas como el RMSE (Root Mean Squared Error) y se analizará la precisión de las predicciones en un conjunto de prueba.

## Resultado Previsto de Aprendizaje

Al finalizar este caso, el estudiante será capaz de:

- Aplicar técnicas de preprocesamiento y transformación de variables para preparar datos tabulares para regresión.
- Implementar y comparar modelos de regresión como Regresión Lineal, Gradient Boosting y K-Nearest Neighbors usando `scikit-learn`.
- Evaluar el desempeño de los modelos de regresión usando métricas de error.

___

## Contexto del Problema

Usted forma parte del equipo de análisis de una plataforma de arriendo de propiedades y se le ha encomendado construir un modelo que permita estimar el precio de arriendo de una propiedad en función de sus características. Esto permitirá generar estimaciones automáticas de precios, apoyar la toma de decisiones de clientes y propietarios, y mejorar la eficiencia del sistema de recomendación. Para lograrlo, deberá preparar los datos, aplicar varios modelos de regresión y seleccionar el que entregue las predicciones más precisas y robustas.

___

## Librerías Requeridas

Iniciamos importando las librerías que usaremos:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

___

## Importando los Datos

Ahora importamos nuestra base de datos de arriendos en Bogotá.

In [ ]:
apt_data = pd.read_csv('data/base_arriendo_bogota.csv')
apt_data

En términos de formato, todas las columnas tienen el formato esperado, a excepción del número de baños. Veamos qué contenidos hay:

In [ ]:
apt_data.Bathrooms.unique()

En efecto, los baños tienen la opción `'5+'` para las propiedades que tienen más de 5 baños. De momento no haremos ninguna corrección.

___

## Análisis Exploratorio de Datos

Demos un vistazo rápido a las estadísticas principales de los datos numéricos.

In [ ]:
apt_data.describe()

Como se observa, tenemos algunas columnas con datos faltantes, pero además, tenemos algunas columnas con valores máximos extremadamente altos.

___

## Preprocesamiento de Datos

Eliminemos los valores con datos faltantes, ya que esto afectaría las predicciones del modelo. La columna con más datos faltantes es la de Administración, esto sugiere que nuestro modelo sólo aplicará a las propiedades que tienen un costo de administración. Hemos dejado por fuera a las propiedades que no pagan este rubro.

In [ ]:
apt_data.dropna(inplace=True)

In [ ]:
apt_data

Ahora usaremos el algoritmo `Isolation Forest` para eliminar el 5% de los valores atípicos de cada una de las columnas en las que notamos que hay valores atípicos

In [ ]:
# columnas en las que hay valores atípicos
filter_columns = ['Rent Price', 'Area (sqm)', 'Area Const (sqm)', 'Admon', 'lon', 'lat']

In [ ]:
# Ahora nos quedamos solo con las columnas numéricas que nos interesan
outlier_predictions = np.array([False] * len(apt_data))  # Inicializamos una lista para las predicciones de outliers

for col in filter_columns:
    #Ahora aplicamos el algoritmo de Isolation Forest
    model = IsolationForest(contamination=0.05, random_state=42) 
    model.fit(apt_data[[col]])  # Ajustamos el modelo a la columna actual

    # Obtenemos la predicción en nuestros datos
    prediction = model.predict(apt_data[[col]])  # Predecimos los outliers para la columna actual
    outlier_predictions = outlier_predictions | (prediction==-1)  # Actualizamos las predicciones de outliers


# I¨dentificamos y eliminamos los outliers
inlier_indices = np.where(outlier_predictions == False)[0] # Obtenemos los índices de los datos normales
cleaned_data = apt_data.iloc[inlier_indices] # Aplicamos una máscara sobre las filas para tener solo los datos normales


Veamos cuántos datos eliminamos.

In [ ]:
print(f"Tamaño dataset original: {apt_data.shape}")
print(f"Tamaño dataset limpio: {cleaned_data.shape}")

Ahora veamos las estadísticas básicas de este dataset limpio.

In [ ]:
cleaned_data.describe()

Aún hay valores extremos altos, veamos el dataset ordenado por precio de arriendo

In [ ]:
cleaned_data.sort_values(by='Rent Price', ascending=False)

Al parecer, los precios de arriendo altos corresponden a propiedades de lujo en las que este costo no es inusual.

En este momento podemos incluir la columna correspondiente al número de baños en nuestro análisis numérico, para esto debemos modificar aquellos valores que no estén en este formato. Para este caso, a aquellos valores de `5+` se les asignará un valor de `6`.

In [ ]:
cleaned_data['Bathrooms'] = cleaned_data['Bathrooms'].replace('5+', 6).astype(int) # Reemplazamos el valor y cambiamos el formato de texto a numérico

El estrato también será incluido como una variable categórica, ya que los números en este caso representan grupos, no realmente una secuencia numérica.

In [ ]:
cleaned_data['estrato'] = cleaned_data['estrato'].astype('category')

Ahora es importante prescindir de las columnas que no le aportarán información relevante a nuestro modelo de regresión. En este caso, las columnas de nombre de la oferta y el link de este. Además, debemos transformar los valores categóricos de manera que el modelo pueda entenderlos, para eso usaremos la técnica `OneHotEncoder` de `scikit-learn`, la cual transforma cada categoría en una nueva columna binaria (0 o 1). Esta codificación es especialmente útil cuando las categorías no tienen un orden lógico.

In [ ]:
# Eliminamos las columnas que no son útiles e identificamos las variables categóricas
categorical_cols = cleaned_data.drop(columns=['Title']).select_dtypes(include=['object','category']).columns

# Creamos el objeto OneHotEncoder
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# Ajustamos y transformamos nuestros datos categóricos
encoded_features = encoder.fit_transform(cleaned_data[categorical_cols])

# Creamos un nuevo dataset con las nuevas columnas codificadas
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_cols))

# Eliminamos las columnas categóricas de nuestro dataset original
clean_data_final = cleaned_data.drop(categorical_cols, axis=1)

# Concatenamos el dataset con las columnas categóricas decodificadas y las columnas numéricas de nuestro dataset original
clean_data_final = pd.concat([clean_data_final.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

Veamos qué obtenemos.

In [ ]:
clean_data_final

Ahora, la base de datos está lista para los modelos predictivos. Es necesario nuevamente eliminar el título de la oferta; además se debe asignar a una nueva variable la columna que queremos predecir, en este caso el precio.

In [ ]:
X = clean_data_final.drop(['Rent Price', 'Title'], axis=1)
y = clean_data_final['Rent Price']

Veamos una matriz de correlación de las columnas de `X`.

In [ ]:
correlation_matrix = X.corr()

# Crear un mapa de calor
plt.figure(figsize=(12, 10))  
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin = -1, vmax = 1)
plt.title('Matrix de correlación de las variables de X')
plt.show()

Ahora veamos la correlación de estas mismas variables con respecto al precio de arriendo.

In [ ]:
# Calcula la correlaciòn entre las columnas de X y Y
correlation_with_y = X.corrwith(y)


plt.figure(figsize=(10, 6))
correlation_with_y.plot(kind='bar')
plt.title('Correlation between X variables and y (Rent Price)')
plt.xlabel('Variables in X')
plt.ylabel('Correlation with y')
plt.xticks(rotation=90, ha='right')  
plt.tight_layout()
plt.show()

___

### Conjuntos de Datos en Machine Learning: Train, Test, y Validate

En Machine Learning, la correcta división y uso de los datos es esencial para garantizar que un modelo generalice bien y no esté sobreajustado a los datos de entrenamiento. Los conjuntos principales que utilizamos son:

---
1.  **Conjunto de Entrenamiento (Train Set)**

    El conjunto de entrenamiento se utiliza para **ajustar el modelo**. Es el conjunto donde el modelo aprende patrones a partir de las características (features) y etiquetas (labels) proporcionadas. Este proceso involucra:

2.  **Conjunto de Validación (Validate Set)**

    El conjunto de validación se utiliza durante el entrenamiento para evaluar el desempeño del modelo en datos que **no se han utilizado para entrenar**.

3.  **Conjunto de Prueba (Test Set)**

    El conjunto de prueba es un conjunto de datos separado que se utiliza **únicamente después de que el modelo ha sido entrenado y validado**.

4.  **Validación Cruzada (Cross-Validation)**

    La validación cruzada es una técnica que permite aprovechar al máximo los datos disponibles, especialmente cuando no hay suficientes datos para separar un conjunto de validación. Funciona dividiendo los datos en múltiples particiones (o "folds"):

    ### Proceso:
    -   Se divide el conjunto de datos en $k$ subconjuntos aproximadamente iguales.
    -   Se entrena el modelo en $k-1$ subconjuntos y se valida en el subconjunto restante.
    -   Este proceso se repite $k$ veces, rotando el subconjunto de validación en cada iteración.
    -   El desempeño del modelo se evalúa como el promedio de las métricas obtenidas en cada iteración.

    ### Ventajas:
    -   Maximiza el uso de los datos disponibles.
    -   Reduce la varianza en las métricas de evaluación, proporcionando estimaciones más estables.


    > **Relación con Train/Validate/Test**:
    > En validación cruzada, los roles de entrenamiento y validación son rotativos. Sin embargo, el conjunto de prueba permanece separado y se utiliza solo al final.

---

Divideremos los datos en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

___

## Gradient Boosting Regression

Gradient Boosting es un método de aprendizaje supervisado usado para problemas de regresión (predecir valores continuos) y también puede adaptarse a clasificación. Combina muchos modelos débiles (normalmente árboles de decisión pequeños) para crear un modelo fuerte.

Funciona de la siguiente manera: 

-   Entrena un modelo simple (como un árbol de decisión pequeño) para predecir el objetivo.
-   Calcula los errores (residuos) entre las predicciones y los valores reales.
-   Entrena un nuevo modelo para predecir esos errores (los residuos).
-   Suma las predicciones del nuevo modelo al modelo anterior.
-   Repite este proceso muchas veces (de forma iterativa), cada vez ajustando el modelo para corregir los errores anteriores.

Ahora un pequeño ejemplo: 

Supón que intentas predecir el precio de una casa:

El primer árbol predice: $300,000 \rightarrow$ el verdadero valor es $320,000 \rightarrow$ el error es $+20,000$.
-   El segundo árbol intenta predecir ese error.
-   El tercer árbol intenta predecir el error que dejó el segundo… y así sucesivamente.
-   Finalmente, sumas todas esas predicciones para obtener una estimación más precisa.



### Ventajas y Desventajas de Gradient Boosting Regression:

**Ventajas:**
-   Suele superar a otros modelos individuales como regresión lineal o árboles simples.
-   Permite diferentes funciones de pérdida, ajuste fino de parámetros, y personalización.
-   Permite interpretar qué variables influyen más en las predicciones.

**Desventajas:**
-   Al ser secuencial, el tiempo de entrenamiento puede ser alto en grandes conjuntos de datos.
-   Requiere ajuste cuidadoso de `n_estimators`, `learning_rate`, `max_depth`, etc.
-   Tiende más al sobreajuste si no se controla bien.

Veamos su implementación en Python.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Iniciamos el modelo
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# Entrenamos el modelo con los valores de entrenamiento seprados
gbr.fit(X_train, y_train)

# Hacemos la predicción primero sobre sobre los datos de entrenamiento
y_train_pred = gbr.predict(X_train)

# Evaluamos el modelo sobre los datos de entrenamiento
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
print(f"Training RMSE: {train_rmse}")


El modelo `GradientBoostRegressor` tiene argumentos `n_estimators`, que hace referencia al número de árboles que se van a construir (iteraciones del boosting), este valor por default es 100. El parámetro `learning_rate` se refiere a cuánto contribuye cada nuevo árbol al modelo final.

Ahora aplicamos el modelo predictivo sobre los datos de prueba.

In [ ]:
y_test_pred = gbr.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
print(f"Test RMSE: {test_rmse}")

Como podemos ver el RMSE (Root Mean Square Error) es bastante alto con respecto a los valores de arriendo. En este caso, podemos aplicar `GridSearchCV` para encontrar los mejores hiperparámetros para `GradientBoostRegressor`. Incluiremos dos argumentos adicionales: `max_depth`, que es la profundidad máxima de cada árbol, y `min_samples_split`, que es el mínimo de muestras para dividir un nodo.

`GridSearchCV` es una herramienta de `scikit-learn` que te permite buscar automáticamente la mejor combinación de hiperparámetros para un modelo de machine learning, usando validación cruzada (cross-validation).

In [ ]:
# Definimos los parámetros para hacer la búsqueda
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.5],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

# Inicializamos el GBR
gbr = GradientBoostingRegressor(random_state=42)

# Inicializamos el GridSearchCV
grid_search = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Ajustamos el modelo GridSearchCV en los datos de entrenamiento
grid_search.fit(X_train, y_train)

# Obtenemos los mejores parámetros
best_params = grid_search.best_params_
print(f"Best hyperparameters: {best_params}")

# Obtenemos el mejor modelo
best_gbr = grid_search.best_estimator_

# Evaluamos el mejor modelo con los datos de prueba
y_pred = best_gbr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE with best hyperparameters: {rmse}")
# Esta línea puede tomar un poco de tiempo

Ahora con esta nueva predicción podemos analizar cuáles son las características con más influencia en el modelo.

In [ ]:
# Feature importance plot
feature_importance = best_gbr.feature_importances_
feature_names = X_test.columns
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), feature_names[sorted_idx])
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.show()

Comparemos gráficamente los datos que el modelo predice con los datos de entrenamiento.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price')
plt.ylabel('Predicted Rent Price')
plt.title('Actual vs. Predicted Rent Price')
plt.show()

Como se observa, las predicciones están distribuidas en una banda muy ancha comparadas con los valores reales. Existen, sin embargo, algunas aproximaciones que podemos hacer para intentar lograr mejroes resultados:
- Transformar la distribución de los valores de arriendo para que sea más simétrica.
- Eliminar variables que no contribuyen al modelo, o que entregan información repetida.

Analicemos un poco la distribución de los valores de arriendo de nuestra base de datos original.

In [ ]:
y.hist(bins = 100)

Estos valores pueden ser transformados para distribuirse de manera más uniforme. Esto se realiza para que el modelo pueda predecirlos de mejor manera, especialmente si se asumen linealidades o normalidades.

In [ ]:
# Inicializamos la transformación con el método Yeo-Johnson (maneja valores positivos y negativos)
pt = PowerTransformer(method='box-cox')

# Ajustamos y transformamos la variable Y
y_transformed = pt.fit_transform(y.values.reshape(-1, 1))

# Veamos cómo quedan los datos transformados
plt.hist(y_transformed, bins=100)
plt.title('Distribution of Transformed Rent Prices')
plt.xlabel('Transformed Rent Price')
plt.ylabel('Frequency')
plt.show()

De acuerdo con los resultados anteriores, algunas variables que podemos eliminar del modelo son:

- Area (alta correlación con el área construida)
- Zonas (la información de las zonas parece no entregar información relevante)

In [ ]:
X.drop(columns=['Area (sqm)','zone_Chapinero','zone_Guaymaral','zone_Noroccidente', 'zone_Norte', 'zone_Occidental'], inplace=True)

Ahora volvemos a dividir nuestros datos y aplicar el modelo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

Ahora corremos de nuevo el gridsearch usando cross-validation

In [ ]:
# Definimos los parámetros para hacer la búsqueda
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.5],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

# Inicializamos el GBR
gbr = GradientBoostingRegressor(random_state=42)

# Inicializamos el GridSearchCV
grid_search = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Ajustamos el modelo GridSearchCV en los datos de entrenamiento
grid_search.fit(X_train, y_train)

# Obtenemos los mejores parámetros
best_params = grid_search.best_params_
print(f"Best hyperparameters: {best_params}")

# Obtenemos el mejor modelo
best_gbr = grid_search.best_estimator_

# Evaluamos el mejor modelo con los datos de prueba
y_pred = best_gbr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE with best hyperparameters: {rmse}")
# Esta línea puede tomar un poco de tiempo

Como podemos ver, los errores son menores al realizar esta transformación. Ahora veamos cuáles son las características con más importancia para el modelo.

In [ ]:
feature_importance = best_gbr.feature_importances_
feature_names = X_test.columns
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), feature_names[sorted_idx])
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.show()

Ahora veamos la comparación de los valores de la predicción y los de prueba transformados.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price (Transformed)')
plt.ylabel('Predicted Rent Price (Transformed)')
plt.title('Actual vs. Predicted Rent Price (Transformed)')
plt.show()

Veamos ahora el ajuste en los datos originales

In [ ]:
# Invertir la transformación de PowerTransformer
y_test_original = pt.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_original = pt.inverse_transform(y_pred.reshape(-1, 1)).flatten()

plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_pred_original)
plt.plot([y_test_original.min(), y_test_original.max()],
         [y_test_original.min(), y_test_original.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price')
plt.ylabel('Predicted Rent Price')
plt.title('Actual vs. Predicted Rent Price (Original Scale)')
plt.show() 

Como se observa, hay aún un error grande en los resultados del ajuste. Para tratar de mejorar, podemos entonces probar con otro tipo de algoritmo

___

## Regresión lineal

Este es un modelo simple que busca ajustar una línea recta (o un plano en dimensiones mayores) que relacione una variable dependiente $y$ con una o más variables independientes $x$.

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n + \varepsilon$$

Se catacteriza por:

-   Estima los coeficientes $\beta$ minimizando la suma de los errores cuadráticos.
-   Cada coeficiente representa el cambio esperado en $Y$ por una unidad de cambio en $X$, manteniendo las demás variables constantes.
-   Asume una forma funcional específica de la relación entre variables.
-   Factores como correlaciones entre variables independientes o valores extremos pueden afectar mucho el rendimiento del modelo.

### Ventajas y Desventajas de la Regresión Lineal:

**Ventajas:**
-   Ideal como modelo base para establecer un punto de comparación.
-   Entrenamiento muy rápido, incluso con grandes volúmenes de datos.
-   Los coeficientes muestran directamente la influencia de cada variable.
-   Computacionalmente muy económico.

**Desventajas:**
-   No se ajusta bien a patrones complejos o no lineales.
-   Un solo valor extremo puede alterar fuertemente la línea de regresión.
-   No siempre se cumplen los supuestos estadísticos requeridos.

Ahora veamos su implementación.

In [ ]:
# Inicializamos el modelo de regresión lineal
linear_regressor = LinearRegression()

# Entrenamos el modelo
linear_regressor.fit(X_train, y_train)

# Hacemos la predicción sobre los datos de prueba
y_pred = linear_regressor.predict(X_test)

# Evaluamos el modelo
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price (Transformed)')
plt.ylabel('Predicted Rent Price (Transformed)')
plt.title('Actual vs. Predicted Rent Price (Linear Regression - Transformed)')
plt.show()

Veamos el resultado en las unidades originales

In [ ]:
# Invertir la transformación de PowerTransformer
y_test_original = pt.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_original = pt.inverse_transform(y_pred.reshape(-1, 1)).flatten()

plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_pred_original)
plt.plot([y_test_original.min(), y_test_original.max()],
         [y_test_original.min(), y_test_original.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price')
plt.ylabel('Predicted Rent Price')
plt.title('Actual vs. Predicted Rent Price (Original Scale)')
plt.show() 

___

## K-Nearest Neighbors (KNN)

KNN (K-Nearest Neighbors) es un algoritmo supervisado que se utiliza para clasificación y regresión. Es un modelo no paramétrico y basado en instancias, lo que significa que no construye un modelo explícito durante el entrenamiento, sino que memoriza los datos y decide en el momento de la predicción.

**¿Cómo funciona?**

-   Guarda todos los datos de entrenamiento.
-   Cuando se le da un nuevo punto para predecir, calcula la distancia entre ese punto y todos los datos de entrenamiento.
-   Selecciona los $k$ puntos más cercanos (vecinos).
-   Devuelve el promedio de los valores de esos vecinos.

* **Distancia entre puntos**: Usualmente se usa la distancia euclidiana

    $$d(x_i, x_j) = \sqrt{ \sum_{k=1}^{n} (x_{ik} - x_{jk})^2 }$$

* **Predicción para la regresión**: Se hace el promedio de los valores de los vecinos más cercanos:

    $$\hat{y} = \frac{1}{K} \sum_{i=1}^{K} y_{(i)}$$

### Ventajas y Desventajas de KNN:

**Ventajas:**
-   No requiere entrenamiento complejo. Solo guarda los datos.
-   Funciona tanto para clasificación como para regresión.
-   No hace suposiciones sobre la distribución de los datos.
-   Puede capturar patrones no lineales si los datos están bien representados.

**Desventajas:**
-   Para cada predicción, debe calcular distancias a todos los datos.
-   Variables con escalas mayores dominan la distancia. Se recomienda usar `StandardScaler`.
-   Puede ser pesado en memoria si el dataset es grande.
-   Vecinos cercanos con ruido pueden influir negativamente.

Veamos cómo funciona.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline

# Pipeline con escalamiento y modelo, es necesario estandarizar los datos ya que el modelo se basa en distancias
modelo = make_pipeline(
    StandardScaler(),               # Estandariza los datos
    KNeighborsRegressor(n_neighbors=5)  # Modelo KNN
)

# Entrenamiento
modelo.fit(X_train, y_train)

# Predicción
y_pred = modelo.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error: {rmse}")

Con `make_pipeline` te aseguras de que el escalamiento se entrene solo con los datos de entrenamiento.

También podemos usar en este caso `GridSearchCV` para estimar el número adecuado de vecinos $k$ en este modelo.

In [ ]:
param_grid = {
    'kneighborsregressor__n_neighbors': [3, 5, 7, 9]
}

grid = GridSearchCV(modelo, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Mejor valor de k:", grid.best_params_)

In [ ]:
modelo = make_pipeline(
    StandardScaler(),               # Estandariza los datos
    KNeighborsRegressor(n_neighbors=7)  # Modelo KNN
)

# Entrenamiento
modelo.fit(X_train, y_train)

# Predicción
y_pred = modelo.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error: {rmse}")

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price (Transformed)')
plt.ylabel('Predicted Rent Price (Transformed)')
plt.title('Actual vs. Predicted Rent Price (KNN - Transformed)')
plt.show()

Veamos los resultados en las unidades originales

In [ ]:
# Invertir la transformación de PowerTransformer
y_test_original = pt.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_original = pt.inverse_transform(y_pred.reshape(-1, 1)).flatten()

plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_pred_original)
plt.plot([y_test_original.min(), y_test_original.max()],
         [y_test_original.min(), y_test_original.max()], 'k--', lw=2)
plt.xlabel('Actual Rent Price')
plt.ylabel('Predicted Rent Price')
plt.title('Actual vs. Predicted Rent Price (Original Scale)')
plt.show() 

___

## Conclusión 

En general, observamos que con las variables a disposición, ninguno de los modelos captura completamente el comportamiento de los precios de arriendo en la ciudad de Bogotá. Definitivamente más información es necesaria, para poder hacer predicciones más eficientes.

La predicción utilizando modelos de regresión lineal, Gradient Boosting y K-Nearest Neighbors permite capturar diferentes relaciones entre las variables y el valor objetivo. El preprocesamiento adecuado de los datos, incluyendo la normalización y codificación, es clave para mejorar el desempeño de los modelos. Además, analizar la importancia de las variables ayuda a entender mejor los factores que influyen en nuestras variables objetivo. Finalmente, comparar las predicciones con datos de prueba es fundamental para seleccionar el modelo más preciso y confiable para la toma de decisiones.

___

### Origen de los datos

-   Precios arriendo Bogotá

___